In [1]:
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.preprocessing import StandardScaler


2026-01-18 14:40:04.855964: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-18 14:40:05.469504: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-18 14:40:07.289038: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
df = pd.read_csv('Dataset/combined_dataset.csv')

In [3]:
X_raw = df.drop(columns=["Label"])

# Проверка на inf
print(np.isinf(X_raw).any())

# Проверка на слишком большие значения
print((np.abs(X_raw) > 1e308).any())

cols_inf = X_raw.columns[np.isinf(X_raw).any()]
cols_big = X_raw.columns[(np.abs(X_raw) > 1e308).any()]

print("inf:", cols_inf)
print("too big:", cols_big)


Source Port          False
Destination Port     False
Protocol             False
Flow Duration        False
Total Fwd Packets    False
                     ...  
dst_ip_c             False
dst_ip_d             False
dst_is_internal      False
dst_subnet_16        False
dst_subnet_24        False
Length: 100, dtype: bool
Source Port          False
Destination Port     False
Protocol             False
Flow Duration        False
Total Fwd Packets    False
                     ...  
dst_ip_c             False
dst_ip_d             False
dst_is_internal      False
dst_subnet_16        False
dst_subnet_24        False
Length: 100, dtype: bool
inf: Index(['Flow Bytes/s', 'Flow Packets/s'], dtype='object')
too big: Index(['Flow Bytes/s', 'Flow Packets/s'], dtype='object')


In [4]:
df = df.replace([np.inf, -np.inf], np.nan)

df['Flow Bytes/s'] = df['Flow Bytes/s'].fillna(df['Flow Bytes/s'].median())
df['Flow Packets/s'] = df['Flow Packets/s'].fillna(df['Flow Packets/s'].median())

In [5]:
df['Flow Bytes/s'] = df['Flow Bytes/s'].clip(
    lower=df['Flow Bytes/s'].quantile(0.001),
    upper=df['Flow Bytes/s'].quantile(0.999)
)

df['Flow Packets/s'] = df['Flow Packets/s'].clip(
    lower=df['Flow Packets/s'].quantile(0.001),
    upper=df['Flow Packets/s'].quantile(0.999)
)

In [6]:
scaler = StandardScaler()
X = scaler.fit_transform(df.drop(columns=["Label"]))
y = df["Label"]

In [7]:
df

,Source Port,Destination Port,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,...,src_subnet_16,src_subnet_24,dst_ip_a,dst_ip_b,dst_ip_c,dst_ip_d,dst_is_internal,dst_subnet_16,dst_subnet_24,Label
0,443.0,54865.0,6.0,3.0,2.0,0.0,12.0,0.0,6.0,6.0,...,26640.0,6820047.0,192.0,168.0,10.0,5.0,1.0,49320.0,12625930.0,0.0
1,80.0,55054.0,6.0,109.0,1.0,1.0,6.0,6.0,6.0,6.0,...,26640.0,6819868.0,192.0,168.0,10.0,5.0,1.0,49320.0,12625930.0,0.0
2,80.0,55055.0,6.0,52.0,1.0,1.0,6.0,6.0,6.0,6.0,...,26640.0,6819868.0,192.0,168.0,10.0,5.0,1.0,49320.0,12625930.0,0.0
3,443.0,46236.0,6.0,34.0,1.0,1.0,6.0,6.0,6.0,6.0,...,26641.0,6820337.0,192.0,168.0,10.0,16.0,1.0,49320.0,12625930.0,0.0
4,443.0,54863.0,6.0,3.0,2.0,0.0,12.0,0.0,6.0,6.0,...,26643.0,6820804.0,192.0,168.0,10.0,5.0,1.0,49320.0,12625930.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2577657,51114.0,53.0,17.0,32215.0,4.0,2.0,112.0,152.0,28.0,28.0,...,49320.0,12625930.0,192.0,168.0,10.0,3.0,1.0,49320.0,12625930.0,0.0
2577658,24054.0,53.0,17.0,324.0,2.0,2.0,84.0,362.0,42.0,42.0,...,49320.0,12625930.0,192.0,168.0,10.0,3.0,1.0,49320.0,12625930.0,0.0
2577659,443.0,58030.0,6.0,82.0,2.0,1.0,31.0,6.0,31.0,0.0,...,6096.0,1560739.0,192.0,168.0,10.0,51.0,1.0,49320.0,12625930.0,0.0
2577660,51694.0,53.0,17.0,1048635.0,6.0,2.0,192.0,256.0,32.0,32.0,...,49320.0,12625930.0,192.0,168.0,10.0,3.0,1.0,49320.0,12625930.0,0.0


In [8]:
df[df["Label"] < 0]

,Source Port,Destination Port,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,...,src_subnet_16,src_subnet_24,dst_ip_a,dst_ip_b,dst_ip_c,dst_ip_d,dst_is_internal,dst_subnet_16,dst_subnet_24,Label


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=["Label"]), df.Label, test_size=0.2, random_state=42
)


In [14]:
from keras.src.optimizers import Adam

model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=((df.columns.__len__() - 1),)),
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(24, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


/home/kwerty/tf_gpu/lib/python3.10/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [11]:
print(tf.config.list_physical_devices('GPU'))


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [12]:
print(np.isnan(X_train).any(), np.isinf(X_train).any())
print(np.isnan(y_train).any(), np.isinf(y_train).any())


Source Port          False
Destination Port     False
Protocol             False
Flow Duration        False
Total Fwd Packets    False
                     ...  
dst_ip_c             False
dst_ip_d             False
dst_is_internal      False
dst_subnet_16        False
dst_subnet_24        False
Length: 100, dtype: bool Source Port          False
Destination Port     False
Protocol             False
Flow Duration        False
Total Fwd Packets    False
                     ...  
dst_ip_c             False
dst_ip_d             False
dst_is_internal      False
dst_subnet_16        False
dst_subnet_24        False
Length: 100, dtype: bool
False False


In [15]:
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=512,
    validation_split=0.2
)


Epoch 1/20
3223/3223 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - accuracy: 0.9303 - loss: 50103.2344 - val_accuracy: 0.9554 - val_loss: 2680.3662
Epoch 2/20
3223/3223 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - accuracy: 0.9770 - loss: 1619.6710 - val_accuracy: 0.9849 - val_loss: 939.5330
Epoch 3/20
3223/3223 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - accuracy: 0.9817 - loss: 681.7707 - val_accuracy: 0.9746 - val_loss: 588.1399
Epoch 4/20
3223/3223 ━━━━━━━━━━━━━━━━━━━━ 15s 5ms/step - accuracy: 0.9831 - loss: 340.2706 - val_accuracy: 0.9813 - val_loss: 251.6760
Epoch 5/20
3223/3223 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - accuracy: 0.9863 - loss: 213.5644 - val_accuracy: 0.9891 - val_loss: 147.9881
Epoch 6/20
3223/3223 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - accuracy: 0.9879 - loss: 147.4278 - val_accuracy: 0.9889 - val_loss: 136.3030
Epoch 7/20
3223/3223 ━━━━━━━━━━━━━━━━━━━━ 15s 5ms/step - accuracy: 0.9891 - loss: 110.7408 - val_accuracy: 0.9938 - val_loss: 81.7810
Epoch 8/20
3223/3223 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/st

KeyboardInterrupt: 

In [16]:
model.evaluate(X_test, y_test)

16111/16111 ━━━━━━━━━━━━━━━━━━━━ 69s 4ms/step - accuracy: 0.9936 - loss: 88.0529


[88.05292510986328, 0.9935988783836365]

In [22]:
# model.save('models/simple_model.keras')
import tensorflow as tf

tf.saved_model.save(model, "saved_model")

INFO:tensorflow:Assets written to: saved_model/assets


INFO:tensorflow:Assets written to: saved_model/assets
